# 7장 (실습 3): RAG 검색에 문서 권한 유지하기

> 책 『AI 보안, 클라우드에서 시작한다』 7장 [실습] 3번을 노트북에서 단계별로 실행하는 버전입니다. 7-2 "모습이 바뀌어도 등급은 유지되어야 한다"와 4-3의 "임베딩되는 순간 권한 정보가 사라진다"를 작은 RAG 검색으로 직접 확인합니다. 소요 시간은 20~30분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 원본 문서의 권한 없이 색인하면 **AI를 거쳐 볼 수 없던 문서를 받아 가는** 것을 확인합니다.
2. 원본 권한을 색인에 함께 옮기고 **검색 단계에서** 역할로 거르면, 같은 질문에도 누가 묻느냐에 따라 프롬프트에 들어가는 문서가 달라지는 것을 확인합니다.
3. 색인은 원본 권한을 **따라갈 뿐**이라는 점, 그리고 원본을 고친 뒤 **색인도 다시 만들어야** 한다는 점을 확인합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | 예정 |
| 1 | 권한 정보 없이 색인하기 | 예정 |
| 2 | 원본 권한을 함께 색인하고 검색 단계에서 거르기 | 예정 |
| 3 | 원본 권한을 고친 뒤 색인을 다시 만들기 | 예정 |
| 4 | 실무에서는 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

## 실습 시나리오

회사가 사내 문서를 검색해 답하는 AI 챗봇을 열었습니다. 임직원과 상주 협력사 직원이 함께 씁니다. 원본 문서 저장소에서는 인사 문서는 인사팀만, 재무 문서는 재무팀만 열 수 있습니다. 그런데 문서를 RAG 색인으로 옮기면서 이 권한도 함께 옮겨졌을까요?

3장에서 본 것처럼 AI에게 "협력사 직원에게는 재무 문서를 보여주지 마"라고 지시하는 것은 통제가 아닙니다. 권한 검사는 AI 바깥, 문서가 프롬프트에 들어가기 전인 **검색 단계**에서 해야 합니다.

```text
 docs.json ──index --no-acl──▶ 색인(본문만)          ┐
 (원본+권한) ─index --acl────▶ 색인(본문+열람 역할)  ┼─▶ [역할로 거르기] ─▶ [유사도 상위 3건] ─▶ 프롬프트 ─▶ LLM
                                                    ┘   ↑ 권한 검사는 여기서 끝난다
```

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Python 3만 있으면 되고 외부 패키지, Docker, API Key가 필요 없습니다.
>
> **2. 노트북 열기**<br>
> 처음이라면 [SETUP.md](../SETUP.md)의 6단계(VS Code와 Jupyter 노트북 연결)를 먼저 한 번 해 둡니다. 그다음에는 아래 두 방법 가운데 하나로 이 노트북을 엽니다.
> - **VS Code**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs && code .`를 실행합니다. 창 왼쪽 아래에 `WSL: Ubuntu-24.04`가 보이면, 왼쪽 탐색기에서 `ch07-zero-trust-lab` 폴더를 펼쳐 이 노트북을 엽니다. 그리고 오른쪽 위 커널 표시를 눌러 **Select Another Kernel...** → **Python Environments...** → `.venv-lab`을 고릅니다. 목록에 `.venv-lab`이 없으면 6단계 ⑥의 2번처럼 `Python: Select Interpreter` → **Enter interpreter path...**로 `~/.venv-lab/bin/python`의 위치를 먼저 알려 줍니다.
> - **웹 브라우저(Jupyter Lab)**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs/ch07-zero-trust-lab && ~/.venv-lab/bin/jupyter lab`을 실행하고, 화면에 나온 `http://localhost:8888/...` 주소를 Windows의 브라우저에서 엽니다.
> - 노트북 파일만 따로 내려받거나 다른 폴더로 옮기지 마세요. 노트북은 자기가 들어 있는 `ch07-zero-trust-lab` 폴더의 실습 파일을 읽습니다. 위 두 방법처럼 저장소 안에서 열면 따로 신경 쓸 것은 없습니다.
>
> **3. 기준 버전**<br>
> - 아래 출력은 2026년 10월 23일 Ubuntu 24.04(WSL2), Python 3.12에서 실행한 결과입니다. 시각, 번호, ID 같은 값은 실행할 때마다 다릅니다. 직접 실행할 때는 출력이 없는 `ch07_lab3_rag_document_acl.ipynb`을 쓰세요.

## 0. 실습 준비와 실습 파일

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 권한 정보 없이 색인하기 | 예정 |
| 2 | 원본 권한을 함께 색인하고 검색 단계에서 거르기 | 예정 |
| 3 | 원본 권한을 고친 뒤 색인을 다시 만들기 | 예정 |
| 4 | 실무에서는 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

In [1]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "python3": "python3 --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

python3    ✅ Python 3.12.3


| 파일 | 역할 |
|---|---|
| `rag-acl/docs.json` | 원본 문서 5건과 문서마다 열람할 수 있는 역할(`allowed`) |
| `rag-acl/rag.py` | 색인 만들기(`index`)와 질문하기(`ask`). 질문과 가까운 문서를 찾아 "프롬프트에 넣을 문서"로 보여 줌 |

> 📌 (참고) `rag.py`는 실제 RAG처럼 문서를 벡터로 바꾸지 않고, 글자 두 개씩 겹치는 정도로 질문과 가까운 문서를 찾습니다. 이 실습에서 보려는 것은 검색 기술이 아니라 **검색 결과가 누구의 권한으로 걸러지는가**입니다. Vector DB를 쓸 때도 원리는 같습니다.

실습 파일을 `~/rag-lab`으로 복사합니다. 3단계에서 `docs.json`을 고치는데, 저장소의 원본은 그대로 두기 위해서입니다.

In [2]:
import shutil
import sys

WORK = Path.home() / "rag-lab"
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(LAB / "rag-acl", WORK)
DOCS, INDEX = WORK / "docs.json", WORK / "index.json"


def rag(*args):
    """rag.py를 실행하고 출력을 돌려줍니다. 터미널의 python3 rag.py ...와 같습니다."""
    return subprocess.run([sys.executable, "rag.py", *args], cwd=WORK, capture_output=True, text=True).stdout


docs = json.loads(DOCS.read_text())
table(["문서", "제목", "열람할 수 있는 역할"], [(d["id"], d["title"], ", ".join(d["allowed"])) for d in docs])

| 문서 | 제목 | 열람할 수 있는 역할 |
|---|---|---|
| D1 | 사내 공지: 하반기 보안 교육 일정 | employee, partner, hr, finance |
| D2 | 협력사 출입·시스템 이용 안내 | employee, partner |
| D3 | [인사] 2026년 임원 연봉 조정안 | hr |
| D4 | [재무] 3분기 협력사 대금 지급 일정 | finance |
| D5 | [구매] 협력사 단가 협상 내부 기준 | employee, partner |

D5는 구매팀이 실수로 협력사(`partner`)에 열어 둔 상태입니다. 역할은 `employee`(임직원), `partner`(협력사), `hr`(인사팀), `finance`(재무팀) 네 가지입니다.

결과를 한눈에 보려고, 역할 4개가 같은 질문 2개를 던졌을 때 프롬프트에 들어가는 문서를 표로 모으는 함수를 만듭니다.

In [3]:
import re

ROLES = ["employee", "partner", "hr", "finance"]
QUESTIONS = ["협력사 대금 지급 일정과 단가 기준 알려줘", "임원 연봉 인상률 알려줘"]


def retrieved(role, question):
    """rag.py ask의 출력에서 문서 ID만 뽑습니다."""
    return re.findall(r"→ (D\d)", rag("ask", role, question))


def matrix(title):
    rows = [(role, *(", ".join(retrieved(role, q)) or "(없음)" for q in QUESTIONS)) for role in ROLES]
    display(Markdown(f"**{title}**"))
    table(["역할", *QUESTIONS], rows)

## 1. 권한 정보 없이 색인하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 권한 정보 없이 색인하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 원본 권한을 함께 색인하고 검색 단계에서 거르기 | 예정 |
| 3 | 원본 권한을 고친 뒤 색인을 다시 만들기 | 예정 |
| 4 | 실무에서는 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

In [4]:
print(rag("index", "--no-acl"))
print(rag("ask", "partner", QUESTIONS[0]))
print(rag("ask", "employee", QUESTIONS[1]))

색인 5건 생성 (권한 정보 없음)

[partner] 협력사 대금 지급 일정과 단가 기준 알려줘
  → D4 [재무] 3분기 협력사 대금 지급 일정  (유사도 0.50)
  → D5 [구매] 협력사 단가 협상 내부 기준  (유사도 0.25)
  → D1 사내 공지: 하반기 보안 교육 일정  (유사도 0.19)

[employee] 임원 연봉 인상률 알려줘
  → D3 [인사] 2026년 임원 연봉 조정안  (유사도 0.56)



협력사 직원의 질문에 재무팀만 볼 수 있는 대금 지급 일정(D4)이, 일반 임직원의 질문에 인사팀만 볼 수 있는 임원 연봉 조정안(D3)이 프롬프트에 들어갑니다. 원본 저장소에서는 열 수 없던 문서를 AI를 거쳐 받아 가는 것입니다.

색인 파일을 열어 보면 이유가 보입니다. 본문만 복사되고 `allowed`가 없습니다.

In [5]:
print(json.dumps(json.loads(INDEX.read_text())[2], ensure_ascii=False, indent=1))

{
 "id": "D3",
 "title": "[인사] 2026년 임원 연봉 조정안",
 "text": "2026년 임원 연봉은 평균 6% 인상합니다. 성과 등급별 인상률과 개인별 조정 금액은 별첨과 같습니다."
}


In [6]:
matrix("권한 정보 없는 색인")

**권한 정보 없는 색인**

| 역할 | 협력사 대금 지급 일정과 단가 기준 알려줘 | 임원 연봉 인상률 알려줘 |
|---|---|---|
| employee | D4, D5, D1 | D3 |
| partner | D4, D5, D1 | D3 |
| hr | D4, D5, D1 | D3 |
| finance | D4, D5, D1 | D3 |

역할과 상관없이 모두 같은 문서를 받습니다. 4-2의 Copilot 과다 권한 사례와 같은 구조이며, 원본 권한은 맞게 되어 있었는데 색인으로 옮기면서 사라졌다는 점이 다릅니다.

## 2. 원본 권한을 함께 색인하고 검색 단계에서 거르기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 권한 정보 없이 색인하기 | ✅ 완료 |
| 2 | 원본 권한을 함께 색인하고 검색 단계에서 거르기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 원본 권한을 고친 뒤 색인을 다시 만들기 | 예정 |
| 4 | 실무에서는 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

In [7]:
print(rag("index", "--acl"))
print(json.dumps(json.loads(INDEX.read_text())[2], ensure_ascii=False, indent=1))

색인 5건 생성 (권한 포함)

{
 "id": "D3",
 "title": "[인사] 2026년 임원 연봉 조정안",
 "text": "2026년 임원 연봉은 평균 6% 인상합니다. 성과 등급별 인상률과 개인별 조정 금액은 별첨과 같습니다.",
 "allowed": [
  "hr"
 ]
}


이번에는 색인의 문서마다 원본 권한(`allowed`)이 붙었습니다. `rag.py`의 `ask()`는 유사도를 계산하기 **전에** 이 값을 보고 거릅니다.

In [8]:
src = (WORK / "rag.py").read_text()
print(src[src.index("    for e in index:"):src.index("        if score > 0.15")])

    for e in index:
        if "allowed" in e and role not in e["allowed"]:
            continue                                 # 검색 단계에서 사용자 권한으로 거른다(AI 바깥의 권한 검사)
        # 유사도 = 질문의 두 글자 조각 중 문서에도 있는 조각의 비율 (0~1)
        score = len(q & grams(e["title"] + e["text"])) / len(q)



In [9]:
matrix("권한을 함께 색인")

**권한을 함께 색인**

| 역할 | 협력사 대금 지급 일정과 단가 기준 알려줘 | 임원 연봉 인상률 알려줘 |
|---|---|---|
| employee | D5, D1, D2 | (없음) |
| partner | D5, D1, D2 | (없음) |
| hr | D1 | D3 |
| finance | D4, D1 | (없음) |

색인에 문서마다 원본 권한을 붙이고, 검색할 때 사용자 역할로 먼저 거르자 D3와 D4는 각 담당 팀에게만 보입니다. 같은 질문이라도 누가 묻느냐에 따라 프롬프트에 들어가는 문서가 달라집니다. 권한 판단이 AI의 추론이 아니라 검색 단계의 확실한 규칙에서 이루어졌습니다.

그런데 협력사 직원에게 **구매팀의 단가 협상 내부 기준(D5)** 은 여전히 보입니다. 원본 문서의 권한부터 협력사에 열려 있었기 때문입니다. 색인은 원본 권한을 그대로 따라갈 뿐, 원본의 실수를 바로잡지는 못합니다.

## 3. 원본 권한을 고친 뒤 색인을 다시 만들기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 권한 정보 없이 색인하기 | ✅ 완료 |
| 2 | 원본 권한을 함께 색인하고 검색 단계에서 거르기 | ✅ 완료 |
| 3 | 원본 권한을 고친 뒤 색인을 다시 만들기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 실무에서는 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

구매팀이 실수를 알아채고 원본 문서(`docs.json`)에서 D5의 열람 권한을 임직원으로 줄였습니다. **색인은 그대로 둔 채** 다시 질문합니다.

In [10]:
docs = json.loads(DOCS.read_text())
next(d for d in docs if d["id"] == "D5")["allowed"] = ["employee"]
DOCS.write_text(json.dumps(docs, ensure_ascii=False, indent=1))
print("D5 원본 권한:", next(d for d in docs if d["id"] == "D5")["allowed"])
print(rag("ask", "partner", QUESTIONS[0]))

D5 원본 권한: ['employee']
[partner] 협력사 대금 지급 일정과 단가 기준 알려줘
  → D5 [구매] 협력사 단가 협상 내부 기준  (유사도 0.25)
  → D1 사내 공지: 하반기 보안 교육 일정  (유사도 0.19)
  → D2 협력사 출입·시스템 이용 안내  (유사도 0.19)



원본은 고쳤지만 색인에는 예전 권한이 남아 있어 D5가 계속 보입니다. 색인을 다시 만들어야 사라집니다.

In [11]:
print(rag("index", "--acl"))
print(rag("ask", "partner", QUESTIONS[0]))

색인 5건 생성 (권한 포함)

[partner] 협력사 대금 지급 일정과 단가 기준 알려줘
  → D1 사내 공지: 하반기 보안 교육 일정  (유사도 0.19)
  → D2 협력사 출입·시스템 이용 안내  (유사도 0.19)



### 3.1. 네 경우를 한 표로

지금까지의 네 경우를 처음부터 다시 재현해, 협력사 직원에게 D4와 D5가 보였는지 표로 만듭니다.

In [12]:
def set_d5(allowed):
    docs = json.loads(DOCS.read_text())
    next(d for d in docs if d["id"] == "D5")["allowed"] = allowed
    DOCS.write_text(json.dumps(docs, ensure_ascii=False, indent=1))


def partner_sees():
    hits = retrieved("partner", QUESTIONS[0])
    return ["보임" if d in hits else "안 보임" for d in ["D4", "D5"]]


rows = []
set_d5(["employee", "partner"]); rag("index", "--no-acl")
rows.append(("권한 정보 없이 색인", *partner_sees(), "색인으로 옮기며 원본 권한이 사라짐"))
rag("index", "--acl")
rows.append(("권한을 함께 색인", *partner_sees(), "원본 권한부터 협력사에 열려 있었음"))
set_d5(["employee"])
rows.append(("원본만 고치고 색인은 그대로", *partner_sees(), "색인에 예전 권한이 남음"))
rag("index", "--acl")
rows.append(("원본을 고치고 다시 색인", *partner_sees(), "원본과 색인의 권한이 일치"))
table(["단계", "D4 재무 대금 일정", "D5 구매 단가 기준", "이유"], rows)

| 단계 | D4 재무 대금 일정 | D5 구매 단가 기준 | 이유 |
|---|---|---|---|
| 권한 정보 없이 색인 | 보임 | 보임 | 색인으로 옮기며 원본 권한이 사라짐 |
| 권한을 함께 색인 | 안 보임 | 보임 | 원본 권한부터 협력사에 열려 있었음 |
| 원본만 고치고 색인은 그대로 | 안 보임 | 보임 | 색인에 예전 권한이 남음 |
| 원본을 고치고 다시 색인 | 안 보임 | 안 보임 | 원본과 색인의 권한이 일치 |

`[표 실습7-1]` 색인 방식에 따라 협력사 직원에게 보인 문서 — 위 표가 실행으로 재현한 결과입니다.

## 4. 실무에서는

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 권한 정보 없이 색인하기 | ✅ 완료 |
| 2 | 원본 권한을 함께 색인하고 검색 단계에서 거르기 | ✅ 완료 |
| 3 | 원본 권한을 고친 뒤 색인을 다시 만들기 | ✅ 완료 |
| 4 | 실무에서는 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

- **권한은 원본에서 관리합니다**: 색인은 원본 권한을 따라갈 뿐입니다. 4-2의 "과다 공유 점검과 권한 정리"처럼 원본 저장소의 권한부터 정리해야 합니다.
- **권한 변경을 색인에 바로 반영합니다**: 원본 권한이 바뀌거나 문서가 삭제되면 Vector DB의 해당 조각도 바로 갱신·삭제되게 합니다(4-3 보안관리 방안 ③·⑤). 하루에 한 번 몰아서 다시 색인하면 그 사이에는 예전 권한이 살아 있습니다(과제 1).
- **역할은 IdP에서 받습니다**: 이 실습에서는 역할을 명령줄로 넘겼지만, 실무에서는 사용자가 직접 정할 수 없도록 IdP가 발급한 토큰의 역할을 씁니다. 실습 2의 Keycloak 역할(employee, partner)이 바로 그 정보입니다.
- **문서 단위보다 더 잘게 나눌 수도 있습니다**: 한 문서 안에서도 일부만 민감하다면, 조각마다 등급을 붙이고 민감한 조각은 마스킹한 뒤 색인합니다(7-2).

### 4.1. 역할을 사용자가 정하면

마지막 항목을 직접 확인합니다. 이 실습의 `rag.py`는 역할을 명령줄 인자로 받으므로, 협력사 직원이 역할 이름만 바꿔 넣으면 그대로 통과합니다.

In [13]:
print(rag("ask", "finance", QUESTIONS[0]))

[finance] 협력사 대금 지급 일정과 단가 기준 알려줘
  → D4 [재무] 3분기 협력사 대금 지급 일정  (유사도 0.50)
  → D1 사내 공지: 하반기 보안 교육 일정  (유사도 0.19)



검색 단계의 필터가 아무리 정확해도, **필터에 넣는 역할을 누가 정하느냐**가 틀리면 소용이 없습니다. 역할은 실습 2의 PEP처럼 서명된 토큰에서 꺼내야 합니다.

## 5. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 권한 정보 없이 색인하기 | ✅ 완료 |
| 2 | 원본 권한을 함께 색인하고 검색 단계에서 거르기 | ✅ 완료 |
| 3 | 원본 권한을 고친 뒤 색인을 다시 만들기 | ✅ 완료 |
| 4 | 실무에서는 | ✅ 완료 |
| 5 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 정리 | 예정 |

### 5.1. 체크포인트

- [ ] 권한 정보 없이 색인했을 때 협력사 직원과 임직원의 질문에 섞여 들어간 문서는 무엇이었나요?
- [ ] 권한 검사가 AI의 추론이 아니라 검색 단계에서 이루어진다는 것을 확인했나요?
- [ ] 권한을 함께 색인해도 막지 못한 문서(D5)가 있었던 이유를 설명할 수 있나요?
- [ ] 원본 권한을 고친 뒤에도 결과가 바뀌지 않았다면 무엇이 빠졌던 것일까요?

### 5.2. 과제 1: 권한 변경을 색인에 바로 반영하기

전체 색인을 다시 만들지 않고, **바뀐 문서만** 색인에 반영하는 `sync_index()`를 완성합니다. 원본 저장소가 "권한이 바뀌었다"거나 "문서가 삭제되었다"는 이벤트를 보내면 호출되는 함수라고 생각하면 됩니다.

- 원본(`docs.json`)과 색인(`index.json`)을 비교해, `allowed`가 다른 문서는 색인의 `allowed`를 원본 값으로 바꿉니다.
- 원본에서 사라진 문서는 색인에서도 지웁니다.
- 바꾼 내용을 목록으로 돌려줍니다.

아래 셀은 D2의 권한을 줄이고 D3를 삭제한 뒤 `sync_index()`를 불러, 협력사와 인사팀의 검색 결과로 확인합니다.

In [14]:
def sync_index():
    """원본과 색인의 차이만 색인에 반영하고, 바꾼 내용을 돌려줍니다."""
    docs = {d["id"]: d for d in json.loads(DOCS.read_text())}
    index = json.loads(INDEX.read_text())
    changes = []
    # TODO 1: index의 각 항목 e에 대해, e["id"]가 docs에 없으면 지우고 changes에 기록합니다.
    # TODO 2: docs[e["id"]]["allowed"]가 e["allowed"]와 다르면 바꾸고 changes에 기록합니다.
    INDEX.write_text(json.dumps(index, ensure_ascii=False, indent=1))
    return changes


# 원본에서 일어난 일: D2를 임직원 전용으로 바꾸고, D3(임원 연봉)를 삭제
docs = json.loads(DOCS.read_text())
next(d for d in docs if d["id"] == "D2")["allowed"] = ["employee"]
docs = [d for d in docs if d["id"] != "D3"]
DOCS.write_text(json.dumps(docs, ensure_ascii=False, indent=1))

print("반영한 변경:", sync_index())
matrix("sync_index() 뒤")

반영한 변경: []


**sync_index() 뒤**

| 역할 | 협력사 대금 지급 일정과 단가 기준 알려줘 | 임원 연봉 인상률 알려줘 |
|---|---|---|
| employee | D5, D1, D2 | (없음) |
| partner | D1, D2 | (없음) |
| hr | D1 | D3 |
| finance | D4, D1 | (없음) |

<details>
<summary>💡 힌트 보기</summary>

```python
kept = []
for e in index:
    if e["id"] not in docs:
        changes.append((e["id"], "삭제"))
        continue
    new = docs[e["id"]]["allowed"]
    if e.get("allowed") != new:
        changes.append((e["id"], f"권한 {e.get('allowed')} → {new}"))
        e["allowed"] = new
    kept.append(e)
index = kept
```

완성하면 협력사의 첫 질문에서 D2가 빠지고, 인사팀의 두 번째 질문에서 D3가 사라져야 합니다. 실제 Vector DB에서는 문서 하나가 여러 조각(chunk)으로 나뉘어 들어가므로, **문서 ID로 모든 조각을 찾아** 메타데이터를 바꾸거나 지울 수 있게 색인할 때부터 문서 ID를 조각마다 붙여 둡니다.
</details>

### 5.3. 과제 2: 재색인 주기 정하기 (서술)

전체 재색인은 하루에 한 번, 밤 2시에 돈다고 합시다. 아래 질문에 답해 보세요.

1. 오전 10시에 인사팀이 문서 권한을 줄였다면, 그 문서는 몇 시간 동안 예전 권한으로 검색될 수 있나요?
2. 과제 1의 즉시 반영이 있어도 전체 재색인이 필요한 이유는 무엇일까요? (이벤트 누락, 원본과 색인의 차이 점검)

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

1.
2.

## 6. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 권한 정보 없이 색인하기 | ✅ 완료 |
| 2 | 원본 권한을 함께 색인하고 검색 단계에서 거르기 | ✅ 완료 |
| 3 | 원본 권한을 고친 뒤 색인을 다시 만들기 | ✅ 완료 |
| 4 | 실무에서는 | ✅ 완료 |
| 5 | 체크포인트와 과제 | ✅ 완료 |
| 6 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [15]:
shutil.rmtree(WORK, ignore_errors=True)
print("실습 폴더 삭제됨")

실습 폴더 삭제됨


---
> 실습 2의 Keycloak 토큰에 담긴 역할을 이 실습의 검색 필터에 넣으면, "누가 묻는가"를 사용자가 바꿀 수 없는 RAG가 됩니다. 8장 실습 3에서는 같은 원리를 Agent의 Tool 호출 검증에 적용합니다.